## 02 — Logistic regression baseline

   Interpretable baseline using the preprocessing decisions from `01_eda` (section 6).
   Metrics: ROC-AUC and PR-AUC on a stratified 80/20 split.

In [0]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
PAST_DUE = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
RANDOM_STATE = 42

**1 - Read CSV and mark missing values**

In [0]:
df = pd.read_csv(PATH, index_col=0)

# Flags first (row-level rules, no statistics -> safe before split)
df["past_due_sentinel"] = df[PAST_DUE].isin([96, 98]).any(axis=1).astype(int)
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)

# Then mark invalid values as missing
df.loc[df["age"] == 0, "age"] = np.nan
df[PAST_DUE] = df[PAST_DUE].where(~df[PAST_DUE].isin([96, 98]))

df[["past_due_sentinel", "income_missing", "dependents_missing"]].sum()

**2 - Stratified train/test split**

In [0]:
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"train: {X_train.shape} | default rate {y_train.mean():.4f}")
print(f"test:  {X_test.shape} | default rate {y_test.mean():.4f}")

**Note:** Default rate is the proportion of customers with target = 1. (same segmentation for train and test)

**3 - Quantile clipper (p99)**

- Create a custom transformer that clips extreme values at the 99th percentile.
- The p99 limit is calculated only on the training set (fit) and then applied to the test set (transform). This way, no information from the test leaks into the model.
- It will be used inside the logistic regression pipeline in the next step.

In [0]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Clip each column at an upper quantile learned on train."""
    def __init__(self, q=0.99):
        self.q = q

    def fit(self, X, y=None):
        self.upper_ = np.nanquantile(X, self.q, axis=0)
        return self

    def transform(self, X):
        return np.minimum(X, self.upper_)

**4 - Logistic regression pipeline**

Build the pipeline in 4 steps, applied in order:
- **impute:** fill missing values with the median from the training set.
- **clip:** trim extreme values at the p99 (99th percentile) of the training set.
- **scale:** standardize the features (mean 0, std 1). 
- **model**: logistic regression.
Train with fit on the training data. Each step learns its parameters using only that data.
Generate the default probability for each test customer.
Calculate two metrics:
- **ROC-AUC:** measures how well the model separates defaulters from non-defaulters. 0.5 is random, 1.0 is perfect.
- **PR-AUC:** measures how well the model identifies defaults, which are rare. The reference for a random model is the default rate itself, 0.0668.

In [0]:
from sklearn.compose import ColumnTransformer

FLAGS = ["past_due_sentinel", "income_missing", "dependents_missing"]
NUMERIC = [c for c in X_train.columns if c not in FLAGS]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clip", QuantileClipper(q=0.99)),
        ("scale", StandardScaler()),
    ]), NUMERIC),
    ("flags", "passthrough", FLAGS),  # binary flags: no clipping or scaling
])

logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000)),  # upper bound
])

logreg.fit(X_train, y_train)
proba = logreg.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, proba):.4f}  (random = {y_test.mean():.4f})")

**5 - Results**

| Metric  | Score  | Random baseline |
|---------|--------|-----------------|
| ROC-AUC | 0.8601 | 0.5000          |
| PR-AUC  | 0.3977 | 0.0668          |

* **ROC-AUC 0.860 (Gini ≈ 0.72):** in pairs of one defaulter and one non-defaulter, the model ranks the defaulter higher about 86% of the time.
* **PR-AUC 0.398:** about 6x the random baseline. Still modest in absolute terms, which is expected with only 6.68% defaults: catching defaulters without flagging good borrowers is the hard part of this problem.
* **Preprocessing:** clipping and scaling apply only to continuous features. Binary flags pass through unchanged, since a p99 cap would erase rare flags such as `past_due_sentinel` (0.18% of rows).
* **Convergence:** the solver converges without warnings. `max_iter=1000` is only an upper bound.

**Takeaway:** a simple, interpretable model already performs well. In `03_lightgbm`, the question is how much a more complex model adds on top of this, and whether the gain justifies losing direct interpretability. PR-AUC is the main number to beat.

In [0]:
rng = np.random.default_rng(RANDOM_STATE)
random_scores = rng.random(len(y_test))
print(f"ROC-AUC: {roc_auc_score(y_test, random_scores):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, random_scores):.4f}")

**Sanity check: random scores**

Random predictions should land near ROC-AUC 0.5 and PR-AUC equal to the default rate.

**6 - ROC and PR curves**

- Calculates the points for both curves for all possible thresholds, using the predicted probabilities (proba) and the actual outcomes (y_test).
- **ROC curve (left):** shows, for each threshold, how many defaults the model catches (Y axis) versus how many good payers it wrongly rejects (X axis). The dashed line is the random model.
- **PR curve (right):** shows, for each threshold, recall (X axis) versus precision (Y axis). The dashed line is the random baseline, at the height of the default rate.


In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(y_test, proba)
precision, recall, _ = precision_recall_curve(y_test, proba)
roc_auc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)
base_rate = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"Logistic regression (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random (AUC = 0.500)")
axes[0].set_xlabel("False positive rate (good borrowers flagged)")
axes[0].set_ylabel("True positive rate (defaults caught)")
axes[0].set_title("ROC curve")
axes[0].legend()

axes[1].plot(recall, precision, label=f"Logistic regression (AP = {pr_auc:.3f})")
axes[1].axhline(base_rate, linestyle="--", color="gray", label=f"Random (AP = {base_rate:.3f})")
axes[1].set_xlabel("Recall (defaults caught)")
axes[1].set_ylabel("Precision (flagged who actually defaulted)")
axes[1].set_title("Precision-recall curve")
axes[1].legend()

plt.tight_layout()
plt.show()

**The area under each curve is the ROC-AUC and the PR-AUC.**

**Reading the curves:**
* **ROC:** flagging 20% of good borrowers already catches about 75% of defaults. Catching the remaining defaults costs increasingly more false positives.
* **PR:** precision is about 60% at 20% recall, 38% at 50% recall and 20% at 80% recall. Catching most defaults means most flagged borrowers would actually pay. The noise near zero recall comes from very few borrowers at the top of the ranking.
* Choosing the threshold is a business decision (cost of a missed default vs. a rejected good borrower), addressed in `04_shap_business`.

**7 - Logistic regression coefficients**

- Get the coefficient that the regression gave for each feature. Since the data was standardized, all coefficients are on the same scale: “effect of increasing by 1 standard deviation”.
- Sign of the coefficient: positive increases the risk of default; negative decreases it.
- Magnitude (absolute value): the larger, the more a feature influences the outcome.
- Also calculate the odds ratio, exp(coefficient), which is easier to read. For example, 1.5 means “raising by 1 standard deviation multiplies the chance of default by 1.5”.
- Show a table and a bar plot sorted by importance.

In [0]:
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coef": logreg.named_steps["model"].coef_[0],
})
coefs["odds_ratio"] = np.exp(coefs["coef"])
coefs = coefs.reindex(coefs["coef"].abs().sort_values(ascending=False).index)

display(coefs.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#D85A30" if c > 0 else "#1D9E75" for c in coefs["coef"]]
ax.barh(coefs["feature"], coefs["coef"], color=colors)
ax.axvline(0, color="gray", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("Coefficient (log-odds: +1 std for continuous, flag on vs. off for binary)")
ax.set_title("Logistic regression coefficients")
plt.tight_layout()
plt.show()

**Databrcks visualization**

In [0]:
   display(coefs)

Databricks visualization. Run in Databricks to view.

**9 - Sentinel group default rate**

In [0]:
y_train.groupby(X_train["past_due_sentinel"]).agg(["mean", "count"])

**10 - Predicted vs actual default rate for the sentinel group**

In [0]:
check = pd.DataFrame({
    "actual": y_train,
    "predicted": logreg.predict_proba(X_train)[:, 1],
    "past_due_sentinel": X_train["past_due_sentinel"],
})
check.groupby("past_due_sentinel")[["actual", "predicted"]].mean().round(3)

The clipper is removing the flag.
Correction: apply imputation, clipping, and standardization only to the continuous features and pass the flags directly to the model.